# Callas — Thai scam-call dataset, fully on Colab

Runs the **whole pipeline** on Colab (no local machine): search → download → transcribe → diarize → segment → clean → prelabel → review → export. All data is written to **Google Drive** so it survives when the Colab session ends.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 is enough).
2. On Hugging Face: create a token and **accept the terms** for `pyannote/speaker-diarization-3.1` AND `pyannote/segmentation-3.0`.
3. Run the cells top to bottom. Every stage is resumable — safe to re-run.

In [ ]:
!nvidia-smi

In [ ]:
# ffmpeg + python deps. transformers<5 emits reliable Whisper timestamps.
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q "transformers<5" "accelerate>=0.30" "pyannote.audio>=3.1" pythainlp python-crfsuite webvtt-py pandas pyyaml tqdm yt-dlp
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# get the code (public repo)
![ -d callas-dataset ] || git clone https://github.com/JoyChanya/callas-dataset.git
%cd callas-dataset
!git pull -q

In [ ]:
# persist all data to Drive: symlink ./data -> MyDrive/callas-data
from google.colab import drive
drive.mount('/content/drive')
import os
DATA = '/content/drive/MyDrive/callas-data'
os.makedirs(DATA, exist_ok=True)
!rm -rf data && ln -s "$DATA" data
print('data ->', os.path.realpath('data'))

In [ ]:
# Hugging Face token (for diarization)
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Paste HF token: ")
from huggingface_hub import login
login(os.environ["HF_TOKEN"])

### (optional) YouTube cookies
If `search`/`download` fails with *"Sign in to confirm you're not a bot"* (can happen on Colab IPs): export a `cookies.txt` from a logged-in browser (a "Get cookies.txt" extension), upload it here, then set `youtube.cookiefile` in `config.yaml`. Skip this cell otherwise.

In [ ]:
# from google.colab import files; files.upload()   # -> uploads cookies.txt
# !sed -i 's|cookiefile: ""|cookiefile: "/content/callas-dataset/cookies.txt"|' config.yaml

### 1–2. Search + download
Edit `config.yaml` (`search.max_per_query`, queries) first if you want. After search you can open `data/meta/videos.csv` from Drive and delete junk rows before downloading.

In [ ]:
!python run_pipeline.py --stage search --max-per-query 50
!python run_pipeline.py --stage download

### 3–4. Transcribe (large Thai model) + diarize — on GPU

In [ ]:
!python run_pipeline.py --stage transcribe --device cuda --model biodatlab/whisper-th-large-v3-combined
!python run_pipeline.py --stage diarize --device cuda

### 5–7. Turns + clean + prelabel

In [ ]:
!python run_pipeline.py --stage segment
!python run_pipeline.py --stage clean
!python run_pipeline.py --stage prelabel
!head -3 data/review/review.csv

### 8–9. Human review → export
Open `MyDrive/callas-data/review/review.csv`, correct the label columns by hand, save as `review_done.csv` in the same folder, then run the export cell.

In [ ]:
!python run_pipeline.py --stage export
!cat data/final/stats.md